# Soumission hors ligne — trois familles fine-tunées

GPU, Internet désactivé. Attacher la compétition et le paquet d’inférence complet, mesuré sur 50–100 études. Les UID sont relus à chaque exécution ; les cinq folds sont moyennés avant les rangs globaux. Sortie : `/kaggle/working/submission.csv`. Les trois études visibles valident seulement l’exécution. Lancer **Save & Run All**, vérifier le résultat, puis Submit to Competition.

In [ ]:
from pathlib import Path
Path('/kaggle/working/submission.csv').unlink(missing_ok=True)
import importlib.util
import json
import subprocess
import sys
BUNDLE_ROOT = None
if BUNDLE_ROOT is None:
    choices = []
    for p in Path('/kaggle/input').rglob('manifest.json'):
        try:
            if json.loads(p.read_text()).get('format') == 'rsna3-full-models-v1':
                choices.append(p.parent)
        except (OSError, ValueError):
            pass
    if len(choices) != 1:
        raise ValueError(f'Choisir BUNDLE_ROOT parmi {choices}')
    BUNDLE_ROOT = choices[0]
BUNDLE_ROOT = Path(BUNDLE_ROOT)
# Optional Python ABI patch; only its shallow dataset metadata is searched.
WHEELS_PATCH_ROOT = None
patch_manifests = list(Path('/kaggle/input').glob('*/wheels_manifest.json'))
patch_manifests += list(Path('/kaggle/input').glob('datasets/*/*/wheels_manifest.json'))
patch_roots = [p.parent for p in patch_manifests
               if json.loads(p.read_text()).get('format') == 'rsna3-offline-wheels-patch-v1']
if len(patch_roots) > 1:
    raise ValueError(f'Attach a single wheels patch: {patch_roots}')
if patch_roots:
    WHEELS_PATCH_ROOT = patch_roots[0]
BOOTSTRAP_ROOT = WHEELS_PATCH_ROOT or BUNDLE_ROOT
module_spec = importlib.util.spec_from_file_location('kaggle_bootstrap', BOOTSTRAP_ROOT / 'kaggle_bootstrap.py')
bootstrap = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(bootstrap)
if WHEELS_PATCH_ROOT is None:
    bootstrap.install(BUNDLE_ROOT)
else:
    bootstrap.install(BUNDLE_ROOT, wheels_root=WHEELS_PATCH_ROOT)
WORK = Path('/kaggle/working')

if json.loads((BUNDLE_ROOT / 'manifest.json').read_text()).get('required_execution_pipeline'):
    raise ValueError('This package requires 04_soumettre_parallele.ipynb; serial submission cannot use its parallel timing profile.')


In [ ]:
# Détection des données de compétition attachées, y compris les nouveaux chemins Kaggle.
COMPETITION_ROOT = None
if COMPETITION_ROOT is None:
    choices = [p.parent for p in Path('/kaggle/input').rglob('test_series.csv')
               if (p.parent / 'test.csv').exists()]
    if len(choices) != 1:
        raise ValueError(f'Choisir COMPETITION_ROOT parmi {choices}')
    COMPETITION_ROOT = choices[0]
COMPETITION_ROOT = Path(COMPETITION_ROOT)

def dicom_directory(split):
    choices = [COMPETITION_ROOT / name for name in (f'{split}_series', f'{split}_images', split)
               if (COMPETITION_ROOT / name).is_dir()]
    if len(choices) != 1:
        raise ValueError(f'Choisir le dossier DICOM {split} parmi {choices}')
    return choices[0]


In [ ]:
PUBLIC_RECIPE = None  # Manifest optionnel d'un socle exact dans son environnement isolé.
CANDIDATE_3PCT = False  # Fixer avant résultats ; socle public 97 % + ensemble propre 3 %.
OPTIONAL_QUOTA_HOURS = None  # Nécessaire uniquement pour admettre la branche optionnelle.
output_csv = WORK / 'submission.csv'
output_csv.unlink(missing_ok=True)  # Une erreur de bootstrap/chemins ne doit pas exposer un vieux CSV.
command = [sys.executable, '-m', 'rsna3', 'submit', '--bundle', str(BUNDLE_ROOT),
           '--test-csv', str(COMPETITION_ROOT / 'test.csv'),
           '--series-csv', str(COMPETITION_ROOT / 'test_series.csv'),
           '--dicom-root', str(dicom_directory('test')), '--work', str(WORK / 'rsna3-test'),
           '--output-csv', str(output_csv), '--max-hours', '8']
if PUBLIC_RECIPE is not None:
    command += ['--public-recipe', str(PUBLIC_RECIPE)]
if CANDIDATE_3PCT:
    command += ['--candidate-3pct']
if OPTIONAL_QUOTA_HOURS is not None:
    command += ['--optional-quota-hours', str(OPTIONAL_QUOTA_HOURS)]
subprocess.run(command, check=True)


In [ ]:
import pandas as pd
submission = pd.read_csv(output_csv)
print(submission.shape)
display(submission.head())
print((WORK / 'rsna3-test/submission_receipt.json').read_text())
